## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [14]:
df = pl.read_csv(
    "data/flixpatrol.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

Rank,Title,Type,Premiere,Genre,Watchtime,Watchtime in Million
f64,str,str,i64,str,str,str
1.0,"""The Night Agent""","""TV Show""",2023,"""Action""","""812,100,000""","""812.1M"""
2.0,"""Ginny & Georgia""","""TV Show""",2021,"""Drama""","""665,100,000""","""665.1M"""
3.0,"""The Glory""","""TV Show""",2022,"""Thriller""","""622,800,000""","""622.8M"""
4.0,"""Wednesday""","""TV Show""",2022,"""Fantasy""","""507,700,000""","""507.7M"""
5.0,"""Queen Charlotte: A Bridgerton …","""TV Show""",2023,"""Drama""","""503,000,000""","""503.0M"""
6.0,"""You""","""TV Show""",2018,"""Crime""","""440,600,000""","""440.6M"""
7.0,"""La Reina del Sur""","""TV Show""",2011,"""Drama""","""429,600,000""","""429.6M"""
8.0,"""Outer Banks""","""TV Show""",2020,"""Drama""","""402,500,000""","""402.5M"""
9.0,"""Ginny & Georgia""","""TV Show""",2021,"""Drama""","""302,100,000""","""302.1M"""


### Retrieve Number of Nulls in Each Feature

In [15]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""Rank""",0
"""Title""",0
"""Type""",0
"""Premiere""",134
"""Genre""",0
"""Watchtime""",0
"""Watchtime in Million""",0


### Retrieve Basic Information About DataFrame

In [16]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
Rank                           | Float64
Title                          | String
Type                           | String
Premiere                       | Int64
Genre                          | String
Watchtime                      | String
Watchtime in Million           | String


### Display Summary Statistics for All Columns

In [17]:
summary = df.describe()
print(summary)

shape: (9, 8)
┌────────────┬─────────────┬────────────┬─────────┬────────────┬─────────┬────────────┬────────────┐
│ statistic  ┆ Rank        ┆ Title      ┆ Type    ┆ Premiere   ┆ Genre   ┆ Watchtime  ┆ Watchtime  │
│ ---        ┆ ---         ┆ ---        ┆ ---     ┆ ---        ┆ ---     ┆ ---        ┆ in Million │
│ str        ┆ f64         ┆ str        ┆ str     ┆ f64        ┆ str     ┆ str        ┆ ---        │
│            ┆             ┆            ┆         ┆            ┆         ┆            ┆ str        │
╞════════════╪═════════════╪════════════╪═════════╪════════════╪═════════╪════════════╪════════════╡
│ count      ┆ 18164.0     ┆ 18164      ┆ 18164   ┆ 18030.0    ┆ 18164   ┆ 18164      ┆ 18164      │
│ null_count ┆ 0.0         ┆ 0          ┆ 0       ┆ 134.0      ┆ 0       ┆ 0          ┆ 0          │
│ mean       ┆ 9126.719335 ┆ null       ┆ null    ┆ 2014.18829 ┆ null    ┆ null       ┆ null       │
│            ┆             ┆            ┆         ┆ 7          ┆         ┆   

### Find Longest Text Length in Each Column

In [18]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

Title,Type,Genre,Watchtime,Watchtime in Million
u32,u32,u32,u32,u32
104,7,15,11,6


### Retrieve Data Types of All Columns

In [19]:
print("Column data types:\n", df.dtypes)

Column data types:
 [Float64, String, String, Int64, String, String, String]


### Count Unique Values in Each Column

In [20]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                         Unique values in Rank : 18164 
                        Unique values in Title : 14509 
                         Unique values in Type : 2     
                     Unique values in Premiere : 74    
                        Unique values in Genre : 29    
                    Unique values in Watchtime : 710   
         Unique values in Watchtime in Million : 710   


### Check Distribution of Numerical Columns

In [21]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

Rank
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ Rank        │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 18164.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 9126.719335 │
│ std        ┆ 5252.511432 │
│ min        ┆ 1.0         │
│ 25%        ┆ 4592.0      │
│ 50%        ┆ 9133.0      │
│ 75%        ┆ 13673.0     │
│ max        ┆ 18214.0     │
└────────────┴─────────────┘ 


Premiere
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ Premiere    │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 18030.0     │
│ null_count ┆ 134.0       │
│ mean       ┆ 2014.188297 │
│ std        ┆ 8.844017    │
│ min        ┆ 1940.0      │
│ 25%        ┆ 2012.0      │
│ 50%        ┆ 2017.0      │
│ 75%        ┆ 2020.0      │
│ max        ┆ 2023.0      │
└────────────┴─────────────┘ 




### List Unique Values For Certain Features

In [22]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: Type (2 unique values)
shape: (2,)
Series: 'Type' [str]
[
	"Movie"
	"TV Show"
]
--------------------------------------------------
Column: Premiere (74 unique values)
shape: (74,)
Series: 'Premiere' [i64]
[
	null
	1940
	1951
	1952
	1953
	1954
	1955
	1956
	1957
	1958
	1960
	1961
	1962
	1963
	1964
	1965
	1966
	1967
	…
	2007
	2008
	2009
	2010
	2011
	2012
	2013
	2014
	2015
	2016
	2017
	2018
	2019
	2020
	2021
	2022
	2023
]
--------------------------------------------------
Column: Genre (29 unique values)
shape: (29,)
Series: 'Genre' [str]
[
	""
	"Action"
	"Adventure"
	"Animation"
	"Biography"
	"Broadcast"
	"Comedy"
	"Concerts"
	"Crime"
	"Documentary"
	"Drama"
	"Fairy Tale"
	"Family"
	"Fantasy"
	"Game-Show"
	"History"
	"Horror"
	"Musical"
	"News"
	"Reality-Show"
	"Romance"
	"Science Fiction"
	"Sports"
	"Stand-Up"
	"Superhero"
	"Talk Show"
	"Thriller"
	"War"
	"Western"
]
--------------------------------------------------
Column: Watchtime (710 unique values)
shape: (710,)
Series: 'Wa

In [23]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

Column: Rank (18164 unique values)
shape: (18_164,)
Series: 'Rank' [f64]
[
	1.0
	2.0
	3.0
	4.0
	5.0
	6.0
	7.0
	8.0
	9.0
	10.0
	11.0
	12.0
	13.0
	14.0
	15.0
	16.0
	17.0
	18.0
	…
	18198.0
	18199.0
	18200.0
	18201.0
	18202.0
	18203.0
	18204.0
	18205.0
	18206.0
	18207.0
	18208.0
	18209.0
	18210.0
	18211.0
	18212.0
	18213.0
	18214.0
]
--------------------------------------------------
Column: Title (14509 unique values)
shape: (14_509,)
Series: 'Title' [str]
[
	"#Alive"
	"#FriendButMarried"
	"#FriendButMarried 2"
	"#Jowable"
	"#LadyRancho"
	"#NoFilter"
	"#SquadGoals"
	"#blackAF"
	"#realityhigh"
	"'71"
	"'Golden Hour' Billkin The Firs…
	"(Un)Well"
	"...altrimenti ci arrabbiamo!"
	"1 Woman 1 Man"
	"10 Cloverfield Lane"
	"10 Days of a Good Man"
	"10 Minute Workouts"
	"10 Minutes Gone"
	…
	"Üç Harfliler: Beddua"
	"Čertoviny"
	"Červený kapitán"
	"Čtyřlístek ve službách krále"
	"Ōoku: The Inner Chambers"
	"Święty interes"
	"Şahane Hayaller"
	"Špindl"
	"Żyć nie umierać"
	"Ženská na vrcholu"
	"Коза